# Notebook 64 v2 — Final Paper Anchor: Structured Interaction Symbolic Field

Regenerated after Notebook 70. Primary basis:

$$[1, c, r, rc, c^3-\alpha c, r^2-\beta, rc^2]$$

Comparison set: `structured_interaction`, `raw_6`, `block_conditioned`.

Outputs: coefficient manifold, hard-block RMSE, universality score, stability, representative reconstruction, final paper verdict.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os, glob, math, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LassoCV, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, LeaveOneOut
from sklearn.decomposition import PCA
try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    Markdown = lambda x: x
np.random.seed(42)
OUTPUT_DIR = "paper64_v2_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True, "grid.alpha": 0.25})

## 1. Load data

In [ ]:
DATA_PATH = None

def autodetect_data_path():
    candidates = []
    patterns = ["*residual*flow*.parquet","*residual*flow*.csv","*governing*flow*.parquet","*governing*flow*.csv","*.parquet","*.csv","*.pkl","*.pickle"]
    for pat in patterns:
        candidates += glob.glob(pat) + glob.glob(os.path.join("data", pat)) + glob.glob(os.path.join("outputs", pat))
    candidates = [c for c in candidates if os.path.isfile(c)]
    return candidates[0] if candidates else None

def synthetic_dataset():
    systems=["entropy","unevenness"]; tasks=["zeta_vs_gue","zeta_vs_poisson"]; forcing_modes=["capacity_gap","feature_gap","condition_gap"]; ks=[3,5,7]; flow_modes=["linear","nonlinear"]
    rows=[]; sample_id=0
    for system in systems:
      for task in tasks:
       for forcing_mode in forcing_modes:
        for k in ks:
         for flow_mode in flow_modes:
          c_grid=np.linspace(-1.25,1.05,42)
          sys_shift=0.06 if system=="entropy" else -0.04
          task_shift=0.05 if task=="zeta_vs_gue" else -0.03
          force_shift={"capacity_gap":0.00,"feature_gap":0.03,"condition_gap":0.08}[forcing_mode]
          k_shift={3:-0.05,5:0.02,7:0.06}[k]
          flow_shift=0.05 if flow_mode=="nonlinear" else -0.02
          nl_gain=1.0 if flow_mode=="nonlinear" else 0.55
          for path_id in range(14):
            r=-0.75+0.10*path_id+0.05*np.sin(0.7*path_id)
            for window_id,c in enumerate(c_grid):
              g=(0.58*np.tanh(1.35*c)+0.42*c-0.78*r+0.20*r**2+nl_gain*0.07*c**2+nl_gain*0.10*r*c-nl_gain*0.025*r**3+sys_shift+task_shift+force_shift+k_shift+flow_shift)
              if forcing_mode=="condition_gap": g+=0.06*np.sin(2.3*c)
              if system=="entropy": g+=0.03*np.cos(1.2*c)
              if task=="zeta_vs_poisson": g-=0.015*c
              if flow_mode=="linear": g+=-0.09*r**2+0.015*c*r
              dc=c_grid[min(window_id+1,len(c_grid)-1)]-c if window_id<len(c_grid)-1 else c-c_grid[max(window_id-1,0)]
              noise=0.012*np.random.randn(); next_r=r+(g+noise)*dc
              rows.append(dict(system=system,task=task,forcing_mode=forcing_mode,k=k,flow_mode=flow_mode,condition_coord=c,residual=r,predicted_flow=g+noise,next_residual=next_r,delta_condition=dc,sample_id=sample_id,path_id=path_id,window_id=window_id))
              r=next_r; sample_id+=1
    return pd.DataFrame(rows)

def load_dataframe(path):
    ext=os.path.splitext(path)[1].lower()
    if ext==".parquet": return pd.read_parquet(path)
    if ext in [".pkl",".pickle"]: return pd.read_pickle(path)
    return pd.read_csv(path)

def align_schema(df):
    aliases={"condition_coord":["condition_coord","c","condition","cond","condition_coordinate"],"residual":["residual","r","resid"],"predicted_flow":["predicted_flow","flow","g","drdc","delta_residual_per_condition"],"next_residual":["next_residual","r_next","next_r"],"delta_condition":["delta_condition","dc","delta_c"],"forcing_mode":["forcing_mode","forcing","forcing_gap","mode"]}
    rename={}
    for canon, alist in aliases.items():
        for a in alist:
            if a in df.columns and a!=canon:
                rename[a]=canon; break
    df=df.rename(columns=rename)
    if "next_residual" not in df.columns and {"residual","predicted_flow","delta_condition"}.issubset(df.columns): df["next_residual"]=df["residual"]+df["predicted_flow"]*df["delta_condition"]
    if "delta_condition" not in df.columns and "condition_coord" in df.columns:
        df=df.sort_values(["condition_coord"]).copy(); df["delta_condition"]=np.gradient(df["condition_coord"].to_numpy())
    defaults={"system":"unknown_system","task":"unknown_task","forcing_mode":"unknown_forcing","k":5,"flow_mode":"unknown_flow","sample_id":np.arange(len(df)),"path_id":0,"window_id":np.arange(len(df))}
    for k,v in defaults.items():
        if k not in df.columns: df[k]=v
    missing=[c for c in ["condition_coord","residual","predicted_flow"] if c not in df.columns]
    if missing: raise ValueError(f"Missing required columns: {missing}")
    return df

DATA_PATH = DATA_PATH or autodetect_data_path()
if DATA_PATH is None:
    df=synthetic_dataset(); data_source="synthetic_fallback"
else:
    df=align_schema(load_dataframe(DATA_PATH)); data_source=DATA_PATH
df=align_schema(df)
print("Data source:", data_source, "rows:", len(df))
display(df.head())

## 2. Basis definitions and fitting helpers

In [ ]:
PRIMARY_BASIS="structured_interaction"
BASIS_NAMES=["structured_interaction","raw_6","block_conditioned"]

def basis_stats(data):
    r=data["residual"].to_numpy(float); c=data["condition_coord"].to_numpy(float)
    return {"alpha_c3_on_c":float(np.sum(c**4)/max(np.sum(c**2),1e-12)),"mean_r2":float(np.mean(r**2))}

def raw_terms(data,stats=None):
    r=data["residual"].to_numpy(float); c=data["condition_coord"].to_numpy(float)
    return {"1":np.ones_like(r),"c":c,"r":r,"c^3":c**3,"r^2":r**2,"r c^2":r*c**2}

def structured_terms(data,stats=None):
    r=data["residual"].to_numpy(float); c=data["condition_coord"].to_numpy(float); stats=stats or basis_stats(data)
    alpha=stats["alpha_c3_on_c"]; beta=stats["mean_r2"]
    return {"1":np.ones_like(r),"c":c,"r":r,"r c":r*c,"c^3_perp_c":c**3-alpha*c,"r^2_centered":r**2-beta,"r c^2":r*c**2}

def compact_interaction_terms(data,stats=None):
    r=data["residual"].to_numpy(float); c=data["condition_coord"].to_numpy(float)
    return {"1":np.ones_like(r),"c":c,"r":r,"r c":r*c,"c^3":c**3,"r^2":r**2}

def nonlinear_interaction_terms(data,stats=None):
    d=structured_terms(data,stats); r=data["residual"].to_numpy(float); d["r^3"]=r**3; return d

def get_basis_dict(data,basis_name,stats=None,flow_mode=None):
    if basis_name=="raw_6": return raw_terms(data,stats)
    if basis_name=="structured_interaction": return structured_terms(data,stats)
    if basis_name=="block_conditioned": return nonlinear_interaction_terms(data,stats) if str(flow_mode)=="nonlinear" else compact_interaction_terms(data,stats)
    raise ValueError(basis_name)

def design_matrix(data,basis_name,stats=None,flow_mode=None,columns=None):
    d=get_basis_dict(data,basis_name,stats,flow_mode); columns=columns or list(d.keys())
    return np.column_stack([d.get(col,np.zeros(len(data))) for col in columns]), columns

def fit_template(sub,basis_name,alpha=1e-6,flow_mode=None):
    stats=basis_stats(sub); X,names=design_matrix(sub,basis_name,stats,flow_mode); y=sub["predicted_flow"].to_numpy(float)
    beta=np.linalg.solve(X.T@X+alpha*np.eye(X.shape[1]),X.T@y); pred=X@beta
    row={"n":len(sub),"template_rmse":math.sqrt(mean_squared_error(y,pred)),"basis_terms":"|".join(names)}
    row.update({f"coef_{n}":float(v) for n,v in zip(names,beta)})
    return beta,pred,row,stats,names

def predict_with_beta(sub,basis_name,beta,stats=None,flow_mode=None,columns=None):
    X,_=design_matrix(sub,basis_name,stats,flow_mode,columns); return X@np.asarray(beta,float)

def trajectory_gap(sub,basis_name,beta_ref,beta_cmp,stats_ref=None,stats_cmp=None,flow_mode=None,columns=None,n_r0=15):
    cmin,cmax=sub["condition_coord"].min(),sub["condition_coord"].max(); rmin,rmax=sub["residual"].min(),sub["residual"].max()
    cgrid=np.linspace(cmin,cmax,40); r0s=np.linspace(np.quantile(sub["residual"],.05),np.quantile(sub["residual"],.95),n_r0)
    flow_cap=max(1.0,2.5*np.quantile(np.abs(sub["predicted_flow"]),.995))
    def integrate(beta,stats,r0):
        r=float(np.clip(r0,rmin,rmax)); traj=[r]
        for i in range(len(cgrid)-1):
            c=cgrid[i]; dc=float(cgrid[i+1]-c); row=pd.DataFrame({"residual":[r],"condition_coord":[c]})
            g=float(np.clip(predict_with_beta(row,basis_name,beta,stats,flow_mode,columns)[0],-flow_cap,flow_cap))
            r=float(np.clip(r+g*dc,rmin-.5,rmax+.5)); traj.append(r)
        return np.array(traj)
    return float(np.mean([math.sqrt(mean_squared_error(integrate(beta_ref,stats_ref,r0),integrate(beta_cmp,stats_cmp,r0))) for r0 in r0s]))

## 3. Coefficient chart helpers

In [ ]:
def build_coef_table(basis_name):
    group_cols=["system","task","forcing_mode","k","flow_mode"]
    rows=[]; subsets={}; stats_map={}; names_map={}
    for vals,sub in df.groupby(group_cols):
        if len(sub)<30: continue
        flow_mode=vals[4]; beta,pred,stats_row,stats,names=fit_template(sub.copy(),basis_name,flow_mode=flow_mode)
        kval=int(vals[3]) if float(vals[3]).is_integer() else vals[3]
        rid=f"{vals[0]}|{vals[1]}|{vals[2]}|k={kval}|{vals[4]}"
        row={"regime_id":rid,"system":vals[0],"task":vals[1],"forcing_mode":vals[2],"k":float(vals[3]),"flow_mode":vals[4]}
        row.update(stats_row); rows.append(row); subsets[rid]=sub.copy(); stats_map[rid]=stats; names_map[rid]=names
    coef_df=pd.DataFrame(rows).reset_index(drop=True); coef_cols=[c for c in coef_df.columns if c.startswith("coef_")]
    if coef_cols: coef_df[coef_cols]=coef_df[coef_cols].fillna(0.0)
    return coef_df,coef_cols,subsets,stats_map,names_map

def build_symbolic_features(df_in,columns=None,reduced_terms=None):
    X=pd.get_dummies(df_in[["forcing_mode","flow_mode","system","task"]],drop_first=False)
    X["k"]=df_in["k"].astype(float).values; X["k2"]=df_in["k"].astype(float).values**2
    ff=df_in["forcing_mode"].astype(str)+"__x__"+df_in["flow_mode"].astype(str); sf=df_in["system"].astype(str)+"__x__"+df_in["forcing_mode"].astype(str)
    X=pd.concat([X,pd.get_dummies(ff,prefix="ff"),pd.get_dummies(sf,prefix="sf"),pd.get_dummies(df_in["forcing_mode"].astype(str),prefix="fk").multiply(df_in["k"].astype(float).to_numpy().reshape(-1,1))],axis=1)
    if reduced_terms is not None: X=X.reindex(columns=reduced_terms,fill_value=0.0)
    if columns is not None: X=X.reindex(columns=columns,fill_value=0.0)
    return X.astype(float)

def term_stability_table(coef_df,coef_cols,n_splits=8,threshold=1e-4):
    splitter=LeaveOneOut() if len(coef_df)<=12 else KFold(n_splits=min(n_splits,len(coef_df)),shuffle=True,random_state=42)
    feats=build_symbolic_features(coef_df).columns.tolist(); counts={coef:{f:0 for f in feats} for coef in coef_cols}; folds=0
    for train_idx,_ in splitter.split(coef_df):
        train=coef_df.iloc[train_idx].reset_index(drop=True); X=build_symbolic_features(train,columns=feats)
        for coef in coef_cols:
            y=train[coef].to_numpy(float); sc=StandardScaler(); Xs=sc.fit_transform(X)
            model=LassoCV(cv=min(5,len(train)),random_state=folds+1,max_iter=30000).fit(Xs,y)
            for f,v in zip(feats,model.coef_):
                if abs(v)>threshold: counts[coef][f]+=1
        folds+=1
    return pd.DataFrame([{"coefficient":coef,"term":f,"frequency":counts[coef][f]/max(folds,1),"count":counts[coef][f],"folds":folds} for coef in coef_cols for f in feats])

def stable_terms_by_coefficient(coef_df,coef_cols,threshold=0.5):
    st=term_stability_table(coef_df,coef_cols); terms={}
    for coef in coef_cols: terms[coef]=st[(st["coefficient"]==coef)&(st["frequency"]>=threshold)]["term"].tolist()
    return terms,st

def predict_pruned_coefficients(train_df,test_df,coef_cols,terms_by_coef):
    Yhat=np.zeros((len(test_df),len(coef_cols)))
    for j,coef in enumerate(coef_cols):
        terms=terms_by_coef.get(coef,[])
        if not terms: Yhat[:,j]=train_df[coef].mean(); continue
        Xtr=build_symbolic_features(train_df,reduced_terms=terms); Xte=build_symbolic_features(test_df,columns=Xtr.columns,reduced_terms=terms)
        y=train_df[coef].to_numpy(float); sc=StandardScaler(); Xtrs=sc.fit_transform(Xtr); Xtes=sc.transform(Xte)
        Yhat[:,j]=Ridge(alpha=1.0).fit(Xtrs,y).predict(Xtes)
    return Yhat

hard_block_masks_template={"k_high":lambda x:x["k"].astype(float)==7.0,"k_low":lambda x:x["k"].astype(float)==3.0,"forcing::condition":lambda x:x["forcing_mode"].astype(str)=="condition_gap","forcing::capacity":lambda x:x["forcing_mode"].astype(str)=="capacity_gap","forcing::feature":lambda x:x["forcing_mode"].astype(str)=="feature_gap","system::entropy":lambda x:x["system"].astype(str)=="entropy","system::unevenness":lambda x:x["system"].astype(str)=="unevenness","flow::linear":lambda x:x["flow_mode"].astype(str)=="linear","flow::nonlinear":lambda x:x["flow_mode"].astype(str)=="nonlinear"}

## 4. Build basis details and coefficient manifold

In [ ]:
basis_details={}
for basis in BASIS_NAMES:
    print("Building:",basis)
    cdf,ccols,subs,stats,names=build_coef_table(basis)
    tbc,stab=stable_terms_by_coefficient(cdf,ccols,threshold=0.5)
    basis_details[basis]={"coef_df":cdf,"coef_cols":ccols,"subsets":subs,"stats":stats,"names":names,"terms_by_coef":tbc,"stability":stab}
primary_coef_df=basis_details[PRIMARY_BASIS]["coef_df"]; primary_coef_cols=basis_details[PRIMARY_BASIS]["coef_cols"]
display(primary_coef_df.head())
primary_coef_df.to_csv(os.path.join(OUTPUT_DIR,"primary_structured_interaction_coefficients.csv"),index=False)

coef_mat=primary_coef_df[primary_coef_cols].to_numpy(float); Z=PCA(n_components=2,random_state=42).fit_transform(StandardScaler().fit_transform(coef_mat))
manifold_df=primary_coef_df[["regime_id","system","task","forcing_mode","k","flow_mode"]].copy(); manifold_df["PC1"]=Z[:,0]; manifold_df["PC2"]=Z[:,1]
manifold_df.to_csv(os.path.join(OUTPUT_DIR,"primary_coefficient_manifold.csv"),index=False)
fig,axes=plt.subplots(1,3,figsize=(15,4))
for ax,col in zip(axes,["forcing_mode","flow_mode","system"]):
    for val,sub in manifold_df.groupby(col): ax.scatter(sub["PC1"],sub["PC2"],label=str(val),s=35)
    ax.set_title(f"Structured manifold by {col}"); ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.legend(fontsize=8)
plt.tight_layout(); fig.savefig(os.path.join(OUTPUT_DIR,"figure1_structured_coefficient_manifold.png"),dpi=220,bbox_inches="tight"); plt.show()

## 5. Hard-block evaluation

In [ ]:
def evaluate_basis(basis_name):
    d=basis_details[basis_name]; coef_df=d["coef_df"]; coef_cols=d["coef_cols"]; rows=[]
    for block,mask_fn in hard_block_masks_template.items():
        mask=mask_fn(coef_df); train=coef_df.loc[~mask].reset_index(drop=True); test=coef_df.loc[mask].reset_index(drop=True)
        if len(test)==0 or len(train)<5: continue
        beta_hat_mat=predict_pruned_coefficients(train,test,coef_cols,d["terms_by_coef"])
        for i,rid in enumerate(test["regime_id"].tolist()):
            sub=d["subsets"][rid]; flow=test.loc[i,"flow_mode"]; cols=[c.replace("coef_","") for c in coef_cols]
            beta_true=test.loc[i,coef_cols].to_numpy(float); beta_hat=beta_hat_mat[i]
            y_true=sub["predicted_flow"].to_numpy(float); y_pred=predict_with_beta(sub,basis_name,beta_hat,stats=d["stats"][rid],flow_mode=flow,columns=cols)
            rows.append({"basis":basis_name,"block":block,"regime_id":rid,"flow_mode":flow,"forcing_mode":test.loc[i,"forcing_mode"],"system":test.loc[i,"system"],"task":test.loc[i,"task"],"k":test.loc[i,"k"],"n_field_terms":len(cols),"n_stable_chart_terms":sum(len(v) for v in d["terms_by_coef"].values()),"coef_rmse":math.sqrt(mean_squared_error(beta_true,beta_hat)),"field_rmse":math.sqrt(mean_squared_error(y_true,y_pred)),"traj_rmse":trajectory_gap(sub,basis_name,beta_true,beta_hat,stats_ref=d["stats"][rid],stats_cmp=d["stats"][rid],flow_mode=flow,columns=cols)})
    return pd.DataFrame(rows)

eval_df=pd.concat([evaluate_basis(b) for b in BASIS_NAMES],ignore_index=True); eval_df.to_csv(os.path.join(OUTPUT_DIR,"hard_block_eval_all_basis.csv"),index=False)
summary_df=eval_df.groupby("basis")[["coef_rmse","field_rmse","traj_rmse","n_field_terms","n_stable_chart_terms"]].mean().reset_index().sort_values("field_rmse")
display(summary_df); summary_df.to_csv(os.path.join(OUTPUT_DIR,"basis_summary.csv"),index=False)
fig,axes=plt.subplots(1,4,figsize=(17,4))
for ax,metric in zip(axes,["coef_rmse","field_rmse","traj_rmse","n_stable_chart_terms"]): ax.bar(summary_df["basis"],summary_df[metric]); ax.set_title(metric); ax.tick_params(axis="x",rotation=25)
plt.tight_layout(); fig.savefig(os.path.join(OUTPUT_DIR,"figure2_basis_summary.png"),dpi=220,bbox_inches="tight"); plt.show()

## 6. Universality, block RMSE, stability, reconstruction

In [ ]:
def universality_scores(eval_df,metric="traj_rmse",tolerance=0.05):
    bs=eval_df.groupby(["block","basis"])[metric].mean().reset_index(); rows=[]
    for block,sub in bs.groupby("block"):
        best=sub[metric].min()
        for _,row in sub.iterrows(): rows.append({"block":block,"basis":row["basis"],"metric":metric,"value":row[metric],"best":best,"within_tolerance":bool(row[metric]<=(1+tolerance)*best),"is_best":bool(np.isclose(row[metric],best) or row[metric]==best)})
    by_block=pd.DataFrame(rows)
    score=by_block.groupby("basis").agg(universality_score=("within_tolerance","mean"),win_rate=("is_best","mean"),mean_metric=("value","mean"),n_blocks=("block","nunique")).reset_index().sort_values(["universality_score","win_rate","mean_metric"],ascending=[False,False,True])
    return score,by_block
U_TOL=0.05; u_score_df,u_block_df=universality_scores(eval_df,"traj_rmse",U_TOL)
display(u_score_df); u_score_df.to_csv(os.path.join(OUTPUT_DIR,"universality_score.csv"),index=False); u_block_df.to_csv(os.path.join(OUTPUT_DIR,"universality_by_block.csv"),index=False)
fig,axes=plt.subplots(1,2,figsize=(12,4)); axes[0].bar(u_score_df["basis"],u_score_df["universality_score"]); axes[0].set_ylim(0,1.05); axes[0].set_title("Universality score"); axes[0].tick_params(axis="x",rotation=25); axes[1].bar(u_score_df["basis"],u_score_df["win_rate"]); axes[1].set_ylim(0,1.05); axes[1].set_title("Block win rate"); axes[1].tick_params(axis="x",rotation=25); plt.tight_layout(); fig.savefig(os.path.join(OUTPUT_DIR,"figure3_universality_and_win_rate.png"),dpi=220,bbox_inches="tight"); plt.show()
block_summary=eval_df.groupby(["block","basis"])[["field_rmse","traj_rmse"]].mean().reset_index(); block_summary.to_csv(os.path.join(OUTPUT_DIR,"block_summary.csv"),index=False)
for metric in ["field_rmse","traj_rmse"]:
    pivot=block_summary.pivot(index="block",columns="basis",values=metric); fig,ax=plt.subplots(figsize=(13,5)); pivot.plot(kind="bar",ax=ax); ax.set_title(f"Hard-block {metric}"); plt.xticks(rotation=35,ha="right"); plt.tight_layout(); fig.savefig(os.path.join(OUTPUT_DIR,f"figure4_hard_block_{metric}.png"),dpi=220,bbox_inches="tight"); plt.show()
stability_all=pd.concat([v["stability"].assign(basis=k) for k,v in basis_details.items()],ignore_index=True); stability_all.to_csv(os.path.join(OUTPUT_DIR,"symbolic_chart_stability_all.csv"),index=False)
stability_summary=stability_all[stability_all["frequency"]>=0.5].groupby("basis").size().reset_index(name="stable_terms").merge(stability_all.groupby("basis")["frequency"].mean().reset_index(name="mean_selection_frequency"),on="basis",how="outer").fillna(0)
display(stability_summary); stability_summary.to_csv(os.path.join(OUTPUT_DIR,"stability_summary.csv"),index=False)
fig,axes=plt.subplots(1,2,figsize=(12,4)); axes[0].bar(stability_summary["basis"],stability_summary["stable_terms"]); axes[0].set_title("Stable symbolic chart terms"); axes[0].tick_params(axis="x",rotation=25); axes[1].bar(stability_summary["basis"],stability_summary["mean_selection_frequency"]); axes[1].set_title("Mean selection frequency"); axes[1].tick_params(axis="x",rotation=25); plt.tight_layout(); fig.savefig(os.path.join(OUTPUT_DIR,"figure5_symbolic_chart_stability.png"),dpi=220,bbox_inches="tight"); plt.show()

## 7. Representative reconstruction and final verdict

In [ ]:
paired=eval_df.pivot_table(index=["block","regime_id"],columns="basis",values="field_rmse").dropna()
if "raw_6" in paired.columns and PRIMARY_BASIS in paired.columns:
    paired["primary_gain"]=paired["raw_6"]-paired[PRIMARY_BASIS]; rep_block,rep_rid=paired.sort_values("primary_gain",ascending=False).index[0]
else: rep_block,rep_rid=eval_df.iloc[0][["block","regime_id"]]
print("Representative:",rep_block,rep_rid)
fig,axes=plt.subplots(1,len(BASIS_NAMES)+1,figsize=(4*(len(BASIS_NAMES)+1),4),sharey=True); sub_ref=basis_details[PRIMARY_BASIS]["subsets"][rep_rid]; y_true=sub_ref["predicted_flow"].to_numpy(float)
for j,panel in enumerate(["empirical"]+BASIS_NAMES):
    ax=axes[j]
    if panel=="empirical": vals=y_true; sub=sub_ref
    else:
        d=basis_details[panel]; cdf=d["coef_df"]; ccols=d["coef_cols"]; mask=hard_block_masks_template[rep_block](cdf); train=cdf.loc[~mask].reset_index(drop=True); test=cdf.loc[mask].reset_index(drop=True); row=test[test["regime_id"]==rep_rid].reset_index(drop=True)
        if len(row)==0: vals=np.full_like(y_true,np.nan); sub=sub_ref
        else:
            beta_hat=predict_pruned_coefficients(train,row,ccols,d["terms_by_coef"])[0]; cols=[c.replace("coef_","") for c in ccols]; vals=predict_with_beta(sub_ref,panel,beta_hat,stats=basis_stats(sub_ref),flow_mode=row.loc[0,"flow_mode"],columns=cols); sub=sub_ref
    sc=ax.scatter(sub["condition_coord"],sub["residual"],c=vals,s=16); ax.set_title(panel); ax.set_xlabel("condition c");
    if j==0: ax.set_ylabel("residual r")
    plt.colorbar(sc,ax=ax,fraction=0.046,pad=0.04)
fig.suptitle(f"Representative reconstruction: {rep_rid}",y=1.03); plt.tight_layout(); fig.savefig(os.path.join(OUTPUT_DIR,"figure6_representative_reconstruction.png"),dpi=220,bbox_inches="tight"); plt.show()

best_field=summary_df.sort_values("field_rmse").iloc[0]; best_traj=summary_df.sort_values("traj_rmse").iloc[0]; best_u=u_score_df.sort_values(["universality_score","win_rate","mean_metric"],ascending=[False,False,True]).iloc[0]
primary_summary=summary_df[summary_df["basis"]==PRIMARY_BASIS].iloc[0]; primary_u=u_score_df[u_score_df["basis"]==PRIMARY_BASIS].iloc[0]; raw_summary=summary_df[summary_df["basis"]=="raw_6"].iloc[0]
if best_u["basis"]==PRIMARY_BASIS: final_verdict="structured_interaction is final paper-facing basis"
elif best_field["basis"]==PRIMARY_BASIS: final_verdict="structured_interaction is final field-RMSE basis; report universality challenger"
else: final_verdict=f"{PRIMARY_BASIS} remains primary interpretive basis, but {best_u['basis']} is strongest universality challenger"
verdict_df=pd.DataFrame([{"primary_basis":PRIMARY_BASIS,"best_field_basis":best_field["basis"],"best_field_rmse":float(best_field["field_rmse"]),"best_traj_basis":best_traj["basis"],"best_traj_rmse":float(best_traj["traj_rmse"]),"best_universality_basis":best_u["basis"],"best_universality_score":float(best_u["universality_score"]),"primary_field_rmse":float(primary_summary["field_rmse"]),"primary_traj_rmse":float(primary_summary["traj_rmse"]),"primary_universality_score":float(primary_u["universality_score"]),"raw_field_rmse":float(raw_summary["field_rmse"]),"raw_traj_rmse":float(raw_summary["traj_rmse"]),"final_verdict":final_verdict}])
display(verdict_df); verdict_df.to_csv(os.path.join(OUTPUT_DIR,"final_paper_verdict.csv"),index=False)
paragraph=f"""## Final structured interaction field\n\nThe regenerated final-paper analysis uses `structured_interaction` as the primary symbolic field basis: `[1, c, r, rc, c^3 - αc, r^2 - β, rc^2]`. Across hard-block validation, the primary basis obtains field RMSE {primary_summary['field_rmse']:.4f}, trajectory RMSE {primary_summary['traj_rmse']:.4f}, and universality score Uτ={primary_u['universality_score']:.3f}. The best universality basis is `{best_u['basis']}` with Uτ={best_u['universality_score']:.3f}. Final verdict: `{final_verdict}`."""
with open(os.path.join(OUTPUT_DIR,"final_structured_interaction_claim.md"),"w",encoding="utf-8") as f: f.write(paragraph+"\n")
display(Markdown(paragraph))
manifest={"data_source":data_source,"synthetic_fallback":bool(DATA_PATH is None),"primary_basis":PRIMARY_BASIS,"basis_names":BASIS_NAMES,"universality_tolerance":U_TOL,"final_verdict":final_verdict,"outputs":sorted(os.listdir(OUTPUT_DIR))}
with open(os.path.join(OUTPUT_DIR,"manifest.json"),"w",encoding="utf-8") as f: json.dump(manifest,f,indent=2)
display(pd.DataFrame({"output_file":sorted(os.listdir(OUTPUT_DIR))}))
print("Saved outputs under:",OUTPUT_DIR)

## Final statement

Notebook 64 v2 is the regenerated paper anchor. It should replace the old Notebook 64 for final figures and narrative.

Recommended next regeneration:

1. Notebook 65 v2 — robustness/shuffle validation for `structured_interaction`
2. Notebook 66 v2 — ML baseline comparison against `structured_interaction`